<a href="https://colab.research.google.com/github/Innovatewithapple/WiKiRagProduction/blob/main/LanggraphWikipediaArticlesRAGWithRedis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain-huggingface langchain_community langgraph pymupdf faiss-cpu

In [ ]:
!pip install wikipedia

In [ ]:
!pip install langchain_nvidia_ai_endpoints

In [ ]:
!pip install redis

In [55]:
pip install duckduckgo-search -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.5 MB/s eta 0:00:00


In [53]:
from langchain_community.document_loaders import WikipediaLoader
from langgraph.graph import StateGraph
from langchain_community.tools import DuckDuckGoSearchResults
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import pymupdf
import re
from nltk.tokenize import sent_tokenize
import nltk
nltk.download('punkt_tab')
from langchain_core.documents import Document
import torch
from sentence_transformers import SentenceTransformer,CrossEncoder
from transformers import AutoTokenizer,AutoModelForCausalLM
from langchain_core.embeddings import Embeddings
from typing import List
import faiss
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from google.colab import userdata
from huggingface_hub import login
from langchain_nvidia_ai_endpoints import ChatNVIDIA
import redis
import hashlib

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
import wikipedia
wikipedia.set_user_agent("MyRAGProject/1.0 (mihirvyasapple@email.com)")

In [ ]:
#---------Encoder Model-------!
encoder_tokenizer = AutoTokenizer.from_pretrained('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)
encoder_Model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5',trust_remote_code=True)

#----------LLm Tokenizer------!
login(token=userdata.get('huggingfaceToken'))
decoder_tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", trust_remote_code=True)
decoder_tokenizer.pad_token = decoder_tokenizer.eos_token

#---------Reranking Model-----!
reranking_model = CrossEncoder('BAAI/bge-reranker-large',device='cuda:0')

In [ ]:
#-------Load the wikipedia articles--------!
articles = [
    "Apollo program",
    "James Webb Space Telescope",
    "Voyager program",
    "International Space Station",
    "Mars rover",
    "Hubble Space Telescope",
    "SpaceX",
    "ISRO",
    "Black hole",
    "Milky Way"
]

docs = []
for article in articles:
  loader = WikipediaLoader(query=article,load_max_docs=1)
  docs.extend(loader.load())

In [ ]:
def createParent_child_chunks(docs,parent_chunk_size,parent_overlap,child_chunk_size,child_overlap):
  chunk_data = []
  for doc in docs:

    #---------Parent Chunking------!
    parent_chunks=[]
    parent_current_chunk=[]
    parent_current_chunk_len_list=[]
    parent_current_chunk_len = 0

    parent_sentences = sent_tokenize(doc.page_content)
    parent_chunk_token_len = [len(decoder_tokenizer.encode(parent_sent,add_special_tokens=False)) for parent_sent in parent_sentences]

    for parent_sent,parent_token in zip(parent_sentences,parent_chunk_token_len):
      if parent_current_chunk_len + parent_token <= parent_chunk_size:
        parent_current_chunk.append(parent_sent)
        parent_current_chunk_len_list.append(parent_token)
        parent_current_chunk_len += parent_token
      else:
        if parent_current_chunk:
          parent_chunks.append(" ".join(parent_current_chunk))
        parent_current_chunk = parent_current_chunk[-parent_overlap:] + [parent_sent]
        parent_current_chunk_len_list = parent_current_chunk_len_list[-parent_overlap:] + [parent_token]
        parent_current_chunk_len = sum(parent_current_chunk_len_list)
    if parent_current_chunk:
        parent_chunks.append(" ".join(parent_current_chunk))


    #-------Child Chunking--------!
    for p_idx,parent_chunk in enumerate(parent_chunks):
      parent_chunk_sentences = sent_tokenize(parent_chunk)
      parent_chunk_sentence_token_len = [len(encoder_tokenizer.encode(parent_chunk_sentence,add_special_tokens=False)) for parent_chunk_sentence in parent_chunk_sentences]

      child_chunks=[]
      child_current_chunk=[]
      child_current_chunk_len_list=[]
      child_current_chunk_len = 0

      for child_sent,child_token in zip(parent_chunk_sentences,parent_chunk_sentence_token_len):
        if child_current_chunk_len + child_token <= child_chunk_size:
          child_current_chunk.append(child_sent)
          child_current_chunk_len_list.append(child_token)
          child_current_chunk_len += child_token

        else:
          if child_current_chunk:
            child_chunks.append(" ".join(child_current_chunk))
          child_current_chunk = child_current_chunk[-child_overlap:] + [child_sent]
          child_current_chunk_len_list = child_current_chunk_len_list[-child_overlap:] + [child_token]
          child_current_chunk_len = sum(child_current_chunk_len_list)
      if child_current_chunk:
          child_chunks.append(" ".join(child_current_chunk))

      #-------chunk data---------!
      for c_idx,child in enumerate(child_chunks):
        chunk_data.append(
            Document(
                page_content=child,
                metadata={
                    'title':doc.metadata['title'],
                    'c_id':f'{p_idx}_{c_idx}',
                    'p_id':p_idx,
                    'parent_text':parent_chunk
                }
            )
        )
  return chunk_data

In [ ]:
all_chunks = createParent_child_chunks(docs=docs,parent_chunk_size=400,parent_overlap=2,child_chunk_size=150,child_overlap=1)

In [ ]:
print(len(all_chunks))

In [ ]:
class PreloadingSentenceEmbeddings(Embeddings):
  def __init__(self,encoder_Model):
    self.encoder_Model = encoder_Model

  def embed_documents(self,text:List[str]) -> List[List[float]]:
    return self.encoder_Model.encode(text,normalize_embeddings=True,batch_size=32,show_progress_bar=True).tolist()

  def embed_query(self,text:str) -> List[float]:
    return self.encoder_Model.encode(text,normalize_embeddings=True).tolist()

embeddings = PreloadingSentenceEmbeddings(encoder_Model=encoder_Model)

In [ ]:
vector_Store = FAISS.from_documents(documents=all_chunks,embedding=embeddings)

In [ ]:
retriever = vector_Store.as_retriever(search_type='similarity',search_kwargs={'k':5})

In [ ]:
reranker = RunnableLambda(Reranking(reranking_model,top_k=3))

In [ ]:
parser = StrOutputParser()

In [130]:
class GraphState(TypedDict):
    question: str
    context: str
    answer: str
    source: str

In [146]:
builder = StateGraph(GraphState)

In [147]:
def retrieve_node(state: GraphState):

    question = state["question"]

    docs = retriever.invoke(question)

    reranked = reranker.invoke({
        "question": question,
        "context": docs
    })

    return {
        "context": reranked["context"],
        "source":"wiki"
    }

In [148]:
def generate_node(state: GraphState):
    print("STATE IN GENERATE:")
    print(state)

    print("QUESTION:", state["question"])
    print("CONTEXT:", state["context"][:500])

    if "source" in state:
        print("SOURCE:", state["source"])
    answer = (
        prompt
        | chat_Model
        | parser
    ).invoke({
        "question": state["question"],
        "context": state["context"]
    })

    return {
        "answer": answer
    }

In [149]:
grader_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a retrieval grader.

Your job is to determine whether the provided context
contains enough information to answer the question.

Rules:
1. Answer ONLY yes or no.
2. If the answer can be reasonably found in the context, say yes.
3. If the context is missing key information, say no.
4. Do not explain.
"""
    ),
    (
        "human",
        """
Question:
{question}

Context:
{context}
"""
    )
])

In [150]:
search_tool = DuckDuckGoSearchResults()

In [151]:
def websearch_node(state: GraphState):

    print("WEB SEARCH NODE CALLED")
    question = state["question"]
    results = search_tool.invoke(question)
    print(results)
    return {
        "context": results,
        "source":"web"
    }

In [152]:
def grade_node(state: GraphState):

    response = (
        grader_prompt
        | chat_Model
        | parser
    ).invoke({
        "question": state["question"],
        "context": state["context"]
    })


    return response.strip().lower()

In [153]:
def route_question(state: GraphState):

    result = grade_node(state)
    print("GRADER:", result)

    if result == "yes":
        return "generate"

    return "websearch"

In [154]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are an experienced wikipedia expert, You are bound to answer the questions from the provided context only. Speak in natural and respective human accent like this is a debate between two humans.

        Rules:
        1. Do NOT use outside knowledge.
        2. Copy numerical and decimal values exactly as written in the context.
        3. Do NOT round, estimate, modify, or infer numbers.
        4. Do NOT explain or expand abbreviations unless explicitly written in the context.
        5. Preserve important entities, task names, benchmark names, and relationships exactly as written in the context.
        6. Include the specific task or subject associated with the answer if present in the context.
        7. Give a clear and natural answer in 2-3 sentences.
        8. Keep the answer grounded strictly in the provided context.
        9. If the exact answer is not explicitly stated but can be reasonably inferred from the context, provide the most likely answer based ONLY on the context.
        10. If the context is incomplete but still provides useful clues, provide the best answer possible and clearly mention any uncertainty.
        11 .Only say "I don't have enough information about this query",
        when the context is completely unrelated or insufficient.
        """
    ),
    (
        "human",
        """
        Context:
        {context}

        Question:
        {question}
        """
    )
])

In [155]:
nvidia_API='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'
base_url='https://integrate.api.nvidia.com/v1/chat/completions'
model='mistralai/mistral-medium-3.5-128b'

In [156]:
chat_Model = ChatNVIDIA(model=model,nvidia_api_key=nvidia_API,temperature=0.1,max_completion_tokens=100,top_p=0.9)

/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Found mistralai/mistral-medium-3.5-128b in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [157]:
builder.add_node(
    "retrieve",
    retrieve_node
)

builder.add_node(
    "generate",
    generate_node
)

builder.add_node(
    "websearch",
    websearch_node
)

builder.add_edge(
    START,
    "retrieve"
)

builder.add_conditional_edges(
    "retrieve",
    route_question,
    {
        "generate": "generate",
        "websearch": "websearch"
    }
)

builder.add_edge(
    "websearch",
    "generate"
)

builder.add_edge(
    "generate",
    END
)

In [158]:
graph = builder.compile()

In [159]:
result = graph.invoke(
    {
        "question": "What did Donald Trump say yesterday?"
    }
)

print(result["answer"])

GRADER: no
WEB SEARCH NODE CALLED
snippet: 4 days ago · President Donald Trump has signed an agreement with Iran that calls for ... said yesterday," Vance said. "And look, I mean, it's very simple. You can't ..., title: WATCH: Vance holds White House briefing after Trump signs Iran war ..., link: https://www.pbs.org/newshour/politics/watch-live-vance-holds-white-house-briefing-after-trump-signs-iran-war-agreement, snippet: 1 day ago · ... say overall criminal offences are at a 25-year low. Trump announced the move in an executive order to rid Memphis of what he called the “tremendous levels ..., title: US News LIVE | Donald Trump Makes Very Big Announcement - YouTube, link: https://www.youtube.com/watch?v=uXc-3tkLnJE, snippet: 6 days ago · President Donald Trump criticized Israeli Prime Minister Benjamin Netanyahu during bilateral meetings at the G7 summit, saying the Israeli prime minister ..., title: President Donald Trump criticized Israeli Prime Minister Benjamin ..., link: https:/

Even Better

Instead of using the same generate node for both paths:

retrieve
├── wiki_generate
└── websearch
      ↓
   web_generate

Create two generators:

Wiki Generator

Strict prompt:

Answer only from context.
No outside knowledge.
Web Generator

Flexible prompt:

Summarize the search results and answer naturally.

That's actually how many production systems are designed.

In [ ]:
from IPython.display import Image, display

display(
    Image(
        graph.get_graph().draw_mermaid_png()
    )
)

In [ ]:
redis_client = redis.Redis(host='localhost',port=6379,decode_responses=True)

In [ ]:
def AskQuestion(question):
  cache_key = hashlib.md5(question.lower().strip().encode()).hexdigest()
  cached_Answer = redis_client.get(cache_key) # or use question string directly
  if cached_Answer:
    print('Cache Hit!!!')
    return cached_Answer

  print('Cache Miss')
  answer = rag_chain.invoke(question)
  redis_client.set(cache_key,answer,ex=3600)
  return answer

In [ ]:
print(redis_client.ping())

In [ ]:
redis_client.flushdb()

In [ ]:
%%time
wikiAnswer = AskQuestion('Who was the first person on the Moon?')
wikiAnswer

In [ ]:
class Reranking:
  def __init__(self,model,top_k):
    self.model = model
    self.top_k = top_k

  def __call__(self,inputs):
    question = inputs['question']
    docs = inputs['context']

    pairs = [(question,doc.page_content) for doc in docs]
    scores = self.model.predict(inputs=pairs)

    for i,doc in enumerate(docs):
      doc.metadata['rerank_score'] = scores[i]

    reranked = sorted(docs,key=lambda x:x.metadata['rerank_score'],reverse=True)
    top_docs = reranked[:self.top_k]

    unique_parents=[]
    seen = set()

    for doc in top_docs:
      parent_id = doc.metadata['p_id']

      if parent_id not in seen:
        unique_parents.append(doc.metadata['parent_text'])

    return {
        'question':question,
        'context':"\n\n".join(unique_parents)
    }

Redis server start

In [ ]:
!apt-get update -qq
!apt-get install redis-server -qq
!redis-server --daemonize yes
!redis-cli ping

In [ ]:
# Apollo Program:
# - Who was first on the Moon?
# - When did Apollo 11 land?

# ISRO:
# - What does ISRO stand for?
# - Which country operates ISRO?

# JWST:
# - What does JWST observe?
# - When was it launched?

# Black Hole:
# - What is a black hole?
# - Why can't light escape it?